# DATA PREPROCESSING

**PURPOSE:** Clean UCI Online Retail II data (541K transations to 6.3K customers)

**OUTPUT** preprocessed_data.csv (customer-level records with Activated target)

## 1. IMPORT LIBRARIES AND LOAD DATASET

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Load UCI Online Retail II dataset
df = pd.read_csv('Online Retail.csv', encoding='ISO-8859-1')
print(f"Raw data loaded: {df.shape[0]:,} transactions, {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

Raw data loaded: 541,909 transactions, 8 columns
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

        InvoiceDate  UnitPrice  CustomerID         Country  
0  2010/12/01 08:26       2.55     17850.0  United Kingdom  
1  2010/12/01 08:26       3.39     17850.0  United Kingdom  
2  2010/12/01 08:26       2.75     17850.0  United Kingdom  
3  2010/12/01 08:26       3.39     17850.0  United Kingdom  
4  2010/12/01 08:26       3.39     17850.0  United Kingdom  


## 2. HANDLE MISSING VALUES

In [112]:
# Remove records with missing CustomerID
print(f"Before: {df.shape[0]:,} records")
df = df.dropna(subset=['CustomerID'])
print(f'After removing records with missing CustomerID: {df.shape[0]:,} records')

# Impute missing descriptions
df['Description'] = df['Description'].fillna('Unnamed Product')

# Check remaining missing values
print(f"Missing values per column:\n{df.isnull().sum()}")

Before: 541,909 records
After removing records with missing CustomerID: 406,829 records
Missing values per column:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


## 3. REMOVE DUPLICATES AND INVALID RECORDS

In [113]:
# Remove duplicate transactions (the ones's with same invoice and product)
df.drop_duplicates(subset=['InvoiceNo', 'StockCode'], inplace=True)
print(f"After removing duplicates: {df.shape[0]:,} records")

# Remove transactions with negative or zero quantity (cancellations or errors)
df = df[df['Quantity'] > 0]

# Remove zero or negative unit prices (errors)
df = df[df['UnitPrice'] > 0]

print(f"After cleaning: {df.shape[0]:,} records")

After removing duplicates: 396,681 records
After cleaning: 387,839 records


## 4. HANDLE OUTLIERS (CAP AT QUANTILE 99)

In [114]:
# Cap Quantity at 99th percentile
q99_qty = df['Quantity'].quantile(0.99)
df['Quantity'] = df['Quantity'].clip(upper=q99_qty)

# Cap Price at 99th percentile
q99_price = df['UnitPrice'].quantile(0.99)
df['UnitPrice'] = df['UnitPrice'].clip(upper=q99_price)

print(f"Outliers capped at Q99 (Quantity: {q99_qty}, Price: {q99_price})")

Outliers capped at Q99 (Quantity: 120.0, Price: 14.95)


In [115]:
## 5. CONVERT DATA TYPES

In [116]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID'] = df['CustomerID'].astype('int64')
df['Quantity'] = df['Quantity'].astype('int64')
df['UnitPrice'] = df['UnitPrice'].astype('float64')

print("✓ Data types converted")
print(df.dtypes)

✓ Data types converted
InvoiceNo                 str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              int64
Country                   str
dtype: object


## 6. AGGREGATE TRANSACTIONS TO CUSTOMER LEVEL

In [121]:
customer_data = df.groupby('CustomerID').agg({
    'InvoiceNo': 'nunique',              # Number of purchases (Frequency)
    'Quantity': 'sum',                   # Total items
    'UnitPrice': 'sum',                      # Total revenue (Customer Lifetime Value)
    'InvoiceDate': ['min', 'max']        # Account age and recency
}).reset_index()

# Flatten column names
customer_data.columns = ['CustomerID', 'Frequency', 'Total_Quantity', 'CLV', 
                         'First_Purchase_Date', 'Last_Purchase_Date']

print(f"Aggregated to {len(customer_data):,} unique customers")
print(customer_data.head())

Aggregated to 4,338 unique customers
   CustomerID  Frequency  Total_Quantity     CLV First_Purchase_Date  \
0       12346          1             120    1.04 2011-01-18 10:01:00   
1       12347          7            2338  481.21 2010-12-07 14:57:00   
2       12348          4            1717   76.83 2010-12-16 19:09:00   
3       12349          1             631  294.05 2011-11-21 09:51:00   
4       12350          1             197   40.25 2011-02-02 16:01:00   

   Last_Purchase_Date  
0 2011-01-18 10:01:00  
1 2011-12-07 15:52:00  
2 2011-09-25 13:13:00  
3 2011-11-21 09:51:00  
4 2011-02-02 16:01:00  


## 7. CALCULATE RECENCY AND CREATE TARGET VARIABLE

In [122]:
# Reference date: last date in dataset
reference_date = df['InvoiceDate'].max()

# Calculate recency (days since last purchase)
customer_data['Recency_Days'] = (reference_date - customer_data['Last_Purchase_Date']).dt.days

# Calculate account age (days since first purchase)
customer_data['Account_Age_Days'] = (reference_date - customer_data['First_Purchase_Date']).dt.days

# Define Activated Target:
# Active = Purchased recently (≤30 days) AND frequency ≥2 times
customer_data['Activated'] = (
    (customer_data['Recency_Days'] <= 30) & 
    (customer_data['Frequency'] >= 2)
).astype(int)

# Check target distribution
activated_count = customer_data['Activated'].sum()
inactive_count = len(customer_data) - activated_count

print(f"Target Distribution:")
print(f"  Activated (1): {activated_count:,} ({activated_count/len(customer_data)*100:.1f}%)")
print(f"  Inactive (0): {inactive_count:,} ({inactive_count/len(customer_data)*100:.1f}%)")
print(customer_data.head())

Target Distribution:
  Activated (1): 1,469 (33.9%)
  Inactive (0): 2,869 (66.1%)
   CustomerID  Frequency  Total_Quantity     CLV First_Purchase_Date  \
0       12346          1             120    1.04 2011-01-18 10:01:00   
1       12347          7            2338  481.21 2010-12-07 14:57:00   
2       12348          4            1717   76.83 2010-12-16 19:09:00   
3       12349          1             631  294.05 2011-11-21 09:51:00   
4       12350          1             197   40.25 2011-02-02 16:01:00   

   Last_Purchase_Date  Recency_Days  Account_Age_Days  Activated  
0 2011-01-18 10:01:00           325               325          0  
1 2011-12-07 15:52:00             1               366          1  
2 2011-09-25 13:13:00            74               357          0  
3 2011-11-21 09:51:00            18                18          0  
4 2011-02-02 16:01:00           309               309          0  


## 8. QUALITY CHECKS

In [123]:
# Verify no missing values
assert customer_data.isnull().sum().sum() == 0, "❌ Missing values detected!"

# Verify target is binary
assert customer_data['Activated'].isin([0, 1]).all(), "❌ Invalid target values!"

# Verify numeric ranges
assert (customer_data['Frequency'] > 0).all(), "❌ Invalid frequency!"
assert (customer_data['CLV'] > 0).all(), "❌ Invalid CLV!"

print("✓ All quality checks passed")

✓ All quality checks passed


## 9. SAVE PREPROCESSED DATA

In [124]:
customer_data.to_csv('preprocessed_data.csv', index=False)
print("✓ Preprocessed data saved to: preprocessed_data.csv")

# Final summary
print(f"\n--- PREPROCESSING SUMMARY ---")
print(f"Input transactions: 541,909")
print(f"After cleaning: {df.shape[0]:,}")
print(f"Customer records: {len(customer_data):,}")
print(f"Activated: {activated_count:,} ({activated_count/len(customer_data)*100:.1f}%)")

✓ Preprocessed data saved to: preprocessed_data.csv

--- PREPROCESSING SUMMARY ---
Input transactions: 541,909
After cleaning: 387,839
Customer records: 4,338
Activated: 1,469 (33.9%)
